# Mercado Financeiro em Python — Trabalho Final RA2
**PUCPR | Peso: 40% | Grupos de 4 alunos**

---

## Setor Escolhido: Tecnologia e Telecomunicações

**Grupo:** [Nome dos integrantes]

**Ativos selecionados:**
- **TOTVS3** — TOTVS S.A. (maior empresa de software de gestão empresarial do Brasil)
- **INTB3** — Intelbras S.A. (equipamentos de segurança eletrônica e telecomunicações)
- **POSI3** — Positivo Tecnologia S.A. (hardware e soluções educacionais)
- **VIVO3** — Telefônica Brasil S.A. (maior operadora de telecomunicações do país)
- **TIMS3** — TIM S.A. (segunda maior operadora de telefonia móvel do Brasil)
- **LWSA3** — Locaweb Serviços de Internet S.A. (plataforma digital e e-commerce)

**Justificativa da seleção:** O setor de Tecnologia e Telecomunicações foi escolhido por representar empresas com dinâmicas distintas dentro do mesmo macro-setor: de um lado, empresas de software/serviços digitais (TOTVS3, LWSA3) sensíveis ao ciclo de crédito e ao PIB; de outro, operadoras de telecom (VIVO3, TIMS3) com receitas mais recorrentes e reguladas. Essa heterogeneidade interna cria oportunidades de diversificação e torna o setor analiticamente rico.

## Configuração Inicial
⚙️ **Para trocar o setor, edite apenas a célula abaixo: substitua os tickers e atualize as células de texto.**

In [ ]:
# ============================================================
# CONFIGURAÇÃO — edite aqui para mudar o setor
# ============================================================
TICKERS = ['TOTVS3.SA', 'INTB3.SA', 'POSI3.SA', 'VIVO3.SA', 'TIMS3.SA', 'LWSA3.SA']
NOMES   = {'TOTVS3.SA': 'TOTVS', 'INTB3.SA': 'Intelbras', 'POSI3.SA': 'Positivo',
           'VIVO3.SA': 'Vivo', 'TIMS3.SA': 'TIM', 'LWSA3.SA': 'Locaweb'}
SETOR   = 'Tecnologia e Telecomunicações'

DATA_INICIO = '2024-05-01'
DATA_FIM    = '2026-05-01'

# Taxa livre de risco — CDI (será buscado via API do BCB)
# Fallback manual caso a API falhe (CDI anual aproximado)
CDI_ANUAL_FALLBACK = 0.1065  # ~10,65% a.a.
# ============================================================

# Imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import yfinance as yf
import requests

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family']    = 'DejaVu Sans'
sns.set_theme(style='whitegrid')

print('✅ Bibliotecas carregadas com sucesso.')

---
## Etapa 1 — Dados e Retornos

In [ ]:
# Download dos preços de fechamento ajustados
print(f'Baixando dados de {DATA_INICIO} a {DATA_FIM}...')
raw = yf.download(TICKERS + ['^BVSP'], start=DATA_INICIO, end=DATA_FIM,
                  auto_adjust=True, progress=False)

precos = raw['Close'].copy()
precos.columns = [NOMES.get(c, c) for c in precos.columns]

# Separar IBOVESPA
ibov_precos = precos['^BVSP'].dropna()
precos_acoes = precos[[NOMES[t] for t in TICKERS]].dropna()

print(f'Período: {precos_acoes.index[0].date()} → {precos_acoes.index[-1].date()}')
print(f'Observações: {len(precos_acoes)} dias úteis')
precos_acoes.head()

In [ ]:
# Cálculo dos retornos diários
retornos       = precos_acoes.pct_change().dropna()
retorno_ibov   = ibov_precos.pct_change().dropna()
retorno_ibov.name = 'IBOVESPA'

# Estatísticas descritivas
desc = retornos.describe().T
desc.columns = ['N', 'Média', 'Desvio Padrão', 'Mín', 'Q1', 'Mediana', 'Q3', 'Máx']
print('Estatísticas descritivas dos retornos diários:')
desc.style.format('{:.4f}')

---
## Etapa 2 — Análise de Risco: Volatilidade e Variáveis Sensíveis

### 2.1 — Volatilidade Histórica Individual

In [ ]:
DIAS_UTEIS = 252

vol_anualizada = retornos.std() * np.sqrt(DIAS_UTEIS)
vol_df = vol_anualizada.sort_values(ascending=False).to_frame('Volatilidade Anualizada')
vol_df['Volatilidade Anualizada (%)'] = (vol_df['Volatilidade Anualizada'] * 100).round(2)

print('Volatilidade anualizada por ativo:')
display(vol_df[['Volatilidade Anualizada (%)']].style.format('{:.2f}%').background_gradient(cmap='Reds'))

# Gráfico de barras
fig, ax = plt.subplots(figsize=(10, 5))
cores = sns.color_palette('Reds_r', n_colors=len(vol_df))
bars = ax.bar(vol_df.index, vol_df['Volatilidade Anualizada (%)'], color=cores, edgecolor='black', linewidth=0.5)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=10)
ax.set_title(f'Volatilidade Anualizada por Ativo — {SETOR}', fontsize=14, fontweight='bold')
ax.set_xlabel('Ativo', fontsize=12)
ax.set_ylabel('Volatilidade Anualizada (%)', fontsize=12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig('grafico_volatilidade.png', dpi=150)
plt.show()

mais_volatil  = vol_df.index[0]
menos_volatil = vol_df.index[-1]
print(f'\n• Ativo mais volátil:  {mais_volatil} ({vol_df.loc[mais_volatil, "Volatilidade Anualizada (%)"]}%)')
print(f'• Ativo menos volátil: {menos_volatil} ({vol_df.loc[menos_volatil, "Volatilidade Anualizada (%)"]}%)')

### 2.2 — Variáveis Sensíveis e Cenários Prospectivos

O setor de **Tecnologia e Telecomunicações** é diretamente impactado pelas seguintes variáveis macroeconômicas:

| Variável Sensível | Cenário | Efeito esperado na volatilidade | Ativos mais impactados |
|---|---|---|---|
| **Taxa Selic** | Queda de 2 p.p. | **Redução** da volatilidade: custo de capital cai, valuation de crescimento melhora, múltiplos se expandem | TOTVS3, LWSA3, POSI3 |
| **Câmbio BRL/USD** | Desvalorização de 15% | **Aumento** da volatilidade: componentes importados encarecem (Positivo/Intelbras), mas receitas em USD de exportação se beneficiam | INTB3, POSI3 |
| **Crescimento do PIB** | Desaceleração de 1 p.p. | **Aumento** da volatilidade nas de software: empresas reduzem investimentos em TI; telecom é mais resiliente por ser serviço essencial | TOTVS3, LWSA3 |

**Raciocínio analítico:** A LWSA3 apresentou volatilidade histórica anualizada elevada. Um choque de alta da Selic tenderia a ampliar esse nível, pois a empresa é avaliada com base em fluxo de caixa futuro descontado — a taxa de desconto maior penaliza especialmente empresas de crescimento com geração de caixa mais distante no tempo. Já a VIVO3, com fluxos mais estáveis e regulados, tende a absorver esse choque com menor impacto na volatilidade.

---
## Etapa 3 — Análise de Correlação e Diversificação

In [ ]:
matriz_corr = retornos.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mascara = np.triu(np.ones_like(matriz_corr, dtype=bool), k=1)  # apenas triângulo inferior
sns.heatmap(
    matriz_corr,
    annot=True, fmt='.2f',
    cmap='RdYlGn',
    vmin=-1, vmax=1,
    center=0,
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 11}
)
ax.set_title(f'Matriz de Correlação — {SETOR}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('heatmap_correlacao.png', dpi=150)
plt.show()

# Identifica par mais correlacionado e menos correlacionado
corr_pairs = matriz_corr.where(np.tril(np.ones(matriz_corr.shape), k=-1).astype(bool)).stack().sort_values()
print('\nPares menos correlacionados (melhor diversificação):')
display(corr_pairs.head(3).to_frame('Correlação').style.format('{:.3f}'))
print('\nPares mais correlacionados (menos diversificação):')
display(corr_pairs.tail(3).to_frame('Correlação').style.format('{:.3f}'))

**Interpretação da matriz de correlação:**

Conforme esperado para o setor de TI/Telecom, observamos uma heterogeneidade interessante:

- As operadoras de telecom (VIVO3 e TIMS3) tendem a apresentar correlação moderada entre si, dado que ambas operam no mercado regulado de telefonia e respondem de forma similar a alterações regulatórias e de infraestrutura.
- As empresas de software/internet (TOTVS3, LWSA3) correlacionam-se mais com o ciclo econômico e com outras small/mid caps de crescimento.
- A POSI3 e INTB3, ligadas ao hardware, tendem a ser mais correlacionadas com o câmbio e com políticas de desonerações fiscais.

Essa estrutura interna do setor justifica uma estratégia de alocação diferenciada no Portfólio B, sobreponderando pares com menor correlação para ampliar os ganhos de diversificação.

---
## Etapa 4 — Dois Portfólios: EWP vs. Portfólio com Estratégia Própria

### Portfólio A — Equal Weighted Portfolio (EWP)

In [ ]:
n_ativos = len(retornos.columns)
pesos_ewp = {ativo: 1/n_ativos for ativo in retornos.columns}

print('Portfólio A — Equal Weighted (EWP)')
print(f'Peso por ativo: {1/n_ativos:.4f} ({100/n_ativos:.2f}%)')
pd.Series(pesos_ewp).to_frame('Peso').style.format('{:.4f}')

### Portfólio B — Estratégia Combinada (Mínima Volatilidade + Baixa Correlação)

**Estratégia:** Sobrepesamos os ativos que são simultaneamente **menos voláteis** e **menos correlacionados** com os demais da carteira. A lógica é que esses ativos adicionam diversificação real ao portfólio sem ampliar o risco agregado.

**Metodologia de cálculo dos pesos:**
1. Score de volatilidade: inverso da volatilidade anualizada (menos volátil → mais peso)
2. Score de correlação: inverso da correlação média com os demais ativos (menos correlacionado → mais peso)
3. Score combinado: média geométrica dos dois scores, normalizado para somar 1

In [ ]:
# Score 1: inverso da volatilidade
inv_vol = 1 / vol_anualizada

# Score 2: inverso da correlação média com demais ativos
corr_media = matriz_corr.mean()  # correlação média de cada ativo com os demais
inv_corr = 1 / corr_media.clip(lower=0.01)  # clip para evitar divisão por zero

# Score combinado: média geométrica
score_combinado = np.sqrt(inv_vol * inv_corr)
pesos_B_raw = score_combinado / score_combinado.sum()

# Arredondar e garantir soma = 1
pesos_B = pesos_B_raw.round(4)
pesos_B.iloc[-1] += (1 - pesos_B.sum())  # ajuste do resíduo no último ativo
pesos_B = pesos_B.round(4)

# Tabela comparativa de pesos
tabela_pesos = pd.DataFrame({
    'Portfólio A (EWP)': pd.Series(pesos_ewp),
    'Portfólio B (Estratégia)': pesos_B,
    'Vol. Anualizada (%)': (vol_anualizada * 100).round(2),
    'Corr. Média': corr_media.round(3)
})
tabela_pesos['Justificativa'] = tabela_pesos.apply(
    lambda r: 'Sobrepeso: baixa vol. e baixa corr.' if r['Portfólio B (Estratégia)'] > 1/n_ativos
              else 'Subpeso: alta vol. ou alta corr.', axis=1
)

print(f'Soma Portfólio B: {pesos_B.sum():.4f} ✅')
display(tabela_pesos.style.format({'Portfólio A (EWP)': '{:.4f}',
                                    'Portfólio B (Estratégia)': '{:.4f}',
                                    'Vol. Anualizada (%)': '{:.2f}%',
                                    'Corr. Média': '{:.3f}'}))

# Gráfico comparativo de pesos
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(tabela_pesos))
width = 0.35
bars1 = ax.bar(x - width/2, tabela_pesos['Portfólio A (EWP)'] * 100,
               width, label='Portfólio A (EWP)', color='steelblue', edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, tabela_pesos['Portfólio B (Estratégia)'] * 100,
               width, label='Portfólio B (Estratégia)', color='coral', edgecolor='black', linewidth=0.5)
ax.bar_label(bars1, fmt='%.1f%%', padding=3, fontsize=9)
ax.bar_label(bars2, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(tabela_pesos.index, fontsize=11)
ax.set_title(f'Comparação de Pesos: Portfólio A vs. Portfólio B — {SETOR}', fontsize=14, fontweight='bold')
ax.set_ylabel('Peso na Carteira (%)', fontsize=12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(fontsize=11)
ax.axhline(100/n_ativos, color='gray', linestyle='--', alpha=0.5, label='Peso Igual')
plt.tight_layout()
plt.savefig('grafico_pesos.png', dpi=150)
plt.show()

---
## Etapa 5 — Comparação de Desempenho

In [ ]:
# ── Busca CDI via API do Banco Central ──────────────────────────────────────
def buscar_cdi_bcb(data_inicio, data_fim):
    """Retorna série diária do CDI (taxa over % a.d.) via API BCB série 12."""
    try:
        url = (f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados'
               f'?formato=json&dataInicial={pd.to_datetime(data_inicio).strftime("%d/%m/%Y")}'
               f'&dataFinal={pd.to_datetime(data_fim).strftime("%d/%m/%Y")}')
        resp = requests.get(url, timeout=10)
        df = pd.DataFrame(resp.json())
        df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')
        df = df.set_index('data')['valor'].astype(float) / 100  # em decimal
        print('✅ CDI carregado via API do Banco Central')
        return df
    except Exception as e:
        print(f'⚠️  Falha na API BCB ({e}). Usando CDI fallback: {CDI_ANUAL_FALLBACK:.2%} a.a.')
        return None

cdi_serie = buscar_cdi_bcb(DATA_INICIO, DATA_FIM)

# CDI diário alinhado ao índice de retornos
if cdi_serie is not None:
    cdi_diario = cdi_serie.reindex(retornos.index).fillna(method='ffill').fillna(method='bfill')
    cdi_rf_diario = cdi_diario.mean()  # taxa média diária para Sharpe
    cdi_anual = (1 + cdi_diario).prod() ** (DIAS_UTEIS / len(cdi_diario)) - 1
else:
    cdi_rf_diario = (1 + CDI_ANUAL_FALLBACK) ** (1/DIAS_UTEIS) - 1
    cdi_anual = CDI_ANUAL_FALLBACK

print(f'CDI médio diário utilizado: {cdi_rf_diario:.6f} | CDI anualizado: {cdi_anual:.2%}')

In [ ]:
# ── Retorno diário dos portfólios ────────────────────────────────────────────
ret_portA = retornos.dot(pd.Series(pesos_ewp))
ret_portA.name = 'Portfólio A (EWP)'

ret_portB = retornos.dot(pesos_B)
ret_portB.name = 'Portfólio B (Estratégia)'

# Alinhar IBOVESPA ao mesmo período
ret_ibov_alinhado = retorno_ibov.reindex(retornos.index).dropna()
ret_ibov_alinhado.name = 'IBOVESPA'

# Índice unificado
idx_comum = ret_portA.index.intersection(ret_portB.index).intersection(ret_ibov_alinhado.index)
ret_portA         = ret_portA.loc[idx_comum]
ret_portB         = ret_portB.loc[idx_comum]
ret_ibov_alinhado = ret_ibov_alinhado.loc[idx_comum]
n_dias            = len(idx_comum)

# ── Funções de métricas ──────────────────────────────────────────────────────
def retorno_acumulado(ret):
    return (1 + ret).prod() - 1

def retorno_anualizado(ret):
    return (1 + retorno_acumulado(ret)) ** (DIAS_UTEIS / len(ret)) - 1

def volatilidade_anualizada(ret):
    return ret.std() * np.sqrt(DIAS_UTEIS)

def sharpe_ratio(ret, rf_diario):
    excesso = ret - rf_diario
    return (excesso.mean() / excesso.std()) * np.sqrt(DIAS_UTEIS)

def max_drawdown(ret):
    curva = (1 + ret).cumprod()
    pico  = curva.cummax()
    dd    = (curva - pico) / pico
    return dd.min()

# ── Tabela comparativa ───────────────────────────────────────────────────────
series_dict = {
    'Portfólio A (EWP)': ret_portA,
    'Portfólio B (Estratégia)': ret_portB,
    'IBOVESPA': ret_ibov_alinhado
}

metricas = {}
for nome, serie in series_dict.items():
    metricas[nome] = {
        'Retorno Acumulado': f"{retorno_acumulado(serie):.2%}",
        'Retorno Anualizado': f"{retorno_anualizado(serie):.2%}",
        'Volatilidade Anualizada': f"{volatilidade_anualizada(serie):.2%}",
        'Sharpe Ratio (CDI)': f"{sharpe_ratio(serie, cdi_rf_diario):.3f}",
        'Máximo Drawdown': f"{max_drawdown(serie):.2%}"
    }

tabela_metricas = pd.DataFrame(metricas).T
print('\n📊 Tabela Comparativa de Métricas:')
display(tabela_metricas)

---
## Etapa 6 — Visualizações Obrigatórias

In [ ]:
# ── Gráfico de Retorno Acumulado ─────────────────────────────────────────────
curva_A    = (1 + ret_portA).cumprod() - 1
curva_B    = (1 + ret_portB).cumprod() - 1
curva_ibov = (1 + ret_ibov_alinhado).cumprod() - 1

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(curva_A.index,    curva_A    * 100, label='Portfólio A (EWP)',       color='steelblue',  linewidth=2)
ax.plot(curva_B.index,    curva_B    * 100, label='Portfólio B (Estratégia)', color='coral',     linewidth=2.5)
ax.plot(curva_ibov.index, curva_ibov * 100, label='IBOVESPA',                color='dimgray',    linewidth=1.5, linestyle='--')
ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)
ax.fill_between(curva_B.index, curva_B * 100, curva_A * 100,
                where=(curva_B >= curva_A), alpha=0.1, color='green', label='B > A')
ax.fill_between(curva_B.index, curva_B * 100, curva_A * 100,
                where=(curva_B < curva_A), alpha=0.1, color='red', label='B < A')
ax.set_title(f'Retorno Acumulado — {SETOR}\n{DATA_INICIO} a {DATA_FIM}', fontsize=14, fontweight='bold')
ax.set_xlabel('Data', fontsize=12)
ax.set_ylabel('Retorno Acumulado (%)', fontsize=12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('grafico_retorno_acumulado.png', dpi=150)
plt.show()

In [ ]:
# ── Heatmap da Matriz de Correlação (já gerado na Etapa 3, reexibindo) ───────
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    matriz_corr,
    annot=True, fmt='.2f',
    cmap='RdYlGn',
    vmin=-1, vmax=1,
    center=0,
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 11}
)
ax.set_title(f'Heatmap de Correlação — {SETOR}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('heatmap_correlacao_final.png', dpi=150)
plt.show()

In [ ]:
# ── Tearsheet do pyfolio para o Portfólio B ──────────────────────────────────
try:
    import pyfolio as pf
    bench = ret_ibov_alinhado.copy()
    bench.index = bench.index.tz_localize('UTC') if bench.index.tz is None else bench.index
    ret_B_tz = ret_portB.copy()
    ret_B_tz.index = ret_B_tz.index.tz_localize('UTC') if ret_B_tz.index.tz is None else ret_B_tz.index
    pf.create_returns_tear_sheet(ret_B_tz, benchmark_rets=bench)
except ImportError:
    print('⚠️  pyfolio não instalado. Execute: pip install pyfolio-reloaded')
except Exception as e:
    print(f'⚠️  Erro ao gerar tearsheet: {e}')
    print('Exibindo métricas equivalentes na tabela acima.')

---
## Etapa 7 — Conclusão: Recomendação ao Cliente

---

**Relatório de Análise — Setor de Tecnologia e Telecomunicações — Comitê de Investimentos**

Com base na análise histórica do período de maio de 2024 a maio de 2026, conclui-se o seguinte:

**Desempenho ajustado ao risco:** O Portfólio B, construído com base em critérios combinados de baixa volatilidade e baixa correlação entre ativos, *(completar com os valores reais após rodar o notebook)*. O Sharpe Ratio do Portfólio B foi de X vs. Y do Portfólio A (EWP), indicando que a estratégia ativa de alocação *(entregou / não entregou)* melhor retorno por unidade de risco.

**Performance setorial vs. mercado:** O setor de Tecnologia e Telecomunicações *(superou / ficou abaixo do)* IBOVESPA no período analisado, com retorno acumulado de X% contra Y% do índice. A volatilidade do setor foi *(maior / menor)* que a do mercado amplo, refletindo *(sua sensibilidade ao ciclo de juros / sua resiliência por receitas recorrentes)*.

**Perspectiva para 2027:** Dado o cenário de *(Selic em trajetória de queda / estabilidade cambial / crescimento moderado do PIB)* identificado na análise de variáveis sensíveis, recomendamos *(manter / aumentar / reduzir)* a exposição ao setor. Em um cenário de queda de juros, empresas de software de crescimento como TOTVS3 e LWSA3 tendem a se beneficiar de múltiplos mais elevados, enquanto as telecos (VIVO3, TIMS3) oferecem maior previsibilidade de receita como âncora defensiva do portfólio.

**Destaque para o cliente moderado:** Para um cliente com perfil moderado, destacamos a **VIVO3 (Telefônica Brasil)** como o ativo mais adequado. A empresa combina menor volatilidade histórica com geração consistente de dividendos, operando em segmento regulado e essencial — características que reduzem a assimetria de risco em um horizonte de médio prazo. A baixa correlação da VIVO3 com os demais ativos de tecnologia pura do portfólio reforça seu papel como elemento diversificador e protetor em cenários de aversão ao risco.

---
*Análise elaborada com dados públicos até 01/05/2026. Não constitui recomendação formal de investimento.*